# Quantization Examples

Loading [AdamLucek/Orpo-Llama-3.2-1B-15k](https://huggingface.co/AdamLucek/Orpo-Llama-3.2-1B-15k) regularly, then quantized to [8-Bit](https://arxiv.org/abs/2208.07339), and quantized to [4-Bit](https://arxiv.org/abs/2305.14314)

In [ ]:
%%capture
!pip install -U transformers bitsandbytes accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

---
# Quantizing and Converting to GGUF Format

[GGUF](https://huggingface.co/docs/hub/gguf) is a specific format developed by Georgi Gerganov in [GGML](https://github.com/ggerganov/ggml?tab=readme-ov-file), specifically for the [llama.cpp](https://github.com/ggerganov/llama.cpp) library. It has become a popular way of storing and distributing language models for a variety of downstream tasks, and comes with built in quantization capabilities.

## Quantization methods

The names of the quantization methods follow the naming convention: "q" + the number of bits + the variant used. K represents the [k-quants quantization method](https://github.com/ggerganov/llama.cpp/pull/1684), a specific quantization method developed for LLMs that handles different parts of the model differently based on their sensitivity to quantization. Here's a simplified writeup of popular methods

* `q2_k`: Uses Q4_K for the attention.vw and feed_forward.w2 tensors, Q2_K for the other tensors.
* `q3_k_l`: Uses Q5_K for the attention.wv, attention.wo, and feed_forward.w2 tensors, else Q3_K
* `q3_k_m`: Uses Q4_K for the attention.wv, attention.wo, and feed_forward.w2 tensors, else Q3_K
* `q3_k_s`: Uses Q3_K for all tensors
* `q4_0`: Original quant method, 4-bit.
* `q4_1`: Higher accuracy than q4_0 but not as high as q5_0. However has quicker inference than q5 models.
* `q4_k_m`: Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K
* `q4_k_s`: Uses Q4_K for all tensors
* `q5_0`: Higher accuracy, higher resource usage and slower inference.
* `q5_1`: Even higher accuracy, resource usage and slower inference.
* `q5_k_m`: Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K
* `q5_k_s`:  Uses Q5_K for all tensors
* `q6_k`: Uses Q8_K for all tensors
* `q8_0`: Almost indistinguishable from float16. High resource use and slow. Not recommended for most users.

We'll be using q4_k_m for 4-bit quantization using the k-quants method, using the outline [here](https://github.com/ggerganov/llama.cpp/blob/master/examples/quantize/README.md)



In [ ]:
# Install llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt

In [ ]:
%cd llama.cpp
!make

In [ ]:
# Download model
%cd .
!curl -sSfL https://hf.co/git-xet/install.sh | sh


In [ ]:

!rm -rf Orpo-Llama-3.2-1B-15k

In [ ]:
!git clone https://huggingface.co/AdamLucek/Orpo-Llama-3.2-1B-15k

In [ ]:
# convert the model to ggml FP16 format
!python convert_hf_to_gguf.py Orpo-Llama-3.2-1B-15k

In [ ]:
# quantize the model to 4-bits (using Q4_K_M method)
!llama.cpp/llama-quantize Orpo-Llama-3.2-1B-15k/Llama-3.2-1B-15k-F16.gguf ./Orpo-Llama-3.2-1B-15k/Orpo-Llama-3.2-1B-15k-Q4_K_M.gguf Q4_K_M

In [ ]:
!cmake -B build
!cmake --build build --config Release


In [ ]:

# quantize the model to 4-bits (using Q4_K_M method)
!./build/bin/llama-quantize Orpo-Llama-3.2-1B-15k/Llama-3.2-1B-15k-F16.gguf ./Orpo-Llama-3.2-1B-15k/Orpo-Llama-3.2-1B-15k-Q4_K_M.gguf Q4_K_M


In [ ]:

# List contents of the model directory to find the generated GGUF file
!ls Orpo-Llama-3.2-1B-15k


## Command Line Testing

`llama.cpp/llama-cli -m Orpo-Llama-3.2-1B-15k/Orpo-Llama-3.2-1B-15k-Q4_K_M.gguf -cnv -p "You are a helpful assistant"`

Model has been uploaded to [AdamLucek/Orpo-Llama-3.2-1B-15k-Q4_K_M-GGUF](https://huggingface.co/AdamLucek/Orpo-Llama-3.2-1B-15k-Q4_K_M-GGUF)

Alternatively, you can use the [gguf-my-repo](https://huggingface.co/spaces/ggml-org/gguf-my-repo) space to quickly convert your HF model to GGUF format with any quantization method.

In [ ]:

!./build/bin/llama-cli -m Orpo-Llama-3.2-1B-15k/Orpo-Llama-3.2-1B-15k-Q4_K_M.gguf -cnv -p "which is the commonly used language in world?"


In [ ]:

!./build/bin/llama-cli -m Llama-3.2-1B-15k-F16.gguf -cnv -p "which is the commonly used language in world?"
